# MESAD + HistoGym — Train on Colab

Run all cells top to bottom.  
At the end the trained model (`dqn_mesad.zip`) is downloaded to your computer.
Then upload it to your Replit project and run `predict.py`.

## 1. Install dependencies

In [ ]:
!pip install -q stable-baselines3 gymnasium opencv-python scikit-learn

## 2. Download & extract the MESAD-real dataset

In [ ]:
import gdown

url = 'https://drive.google.com/file/d/1XJcHoLDqy6Ck70e8PADVfuN5SedJ9AIZ/view'
gdown.download(url, output='data.zip', quiet=False, fuzzy=True)

In [ ]:
import zipfile

with zipfile.ZipFile('/content/data.zip', 'r') as z:
    z.extractall('/content')

TRAIN_IMAGES      = '/content/mesad-real/train/images'
TRAIN_ANNOTATIONS = '/content/mesad-real/train/annotations'
VAL_IMAGES        = '/content/mesad-real/val/images'
VAL_ANNOTATIONS   = '/content/mesad-real/val/annotations'

## 3. Copy the HistoGym-style environment code into Colab

These are the same files that live in `mesad_histogym/` in your Replit project.
We write them here so the notebook is self-contained.

In [ ]:
import os
os.makedirs('/content/mesad_histogym/utils', exist_ok=True)

In [ ]:
%%writefile /content/mesad_histogym/utils/__init__.py
# utils package

In [ ]:
%%writefile /content/mesad_histogym/utils/mesad_annotation.py
"""
MESAD Annotation Parser
Reads TSV annotation files from the MESAD dataset.
"""

import os
import csv
import numpy as np


def _sniff_columns(header):
    header_lc = [h.lower().strip() for h in header]
    has_x1 = any(c in header_lc for c in ('x1', 'xmin', 'left'))
    has_y1 = any(c in header_lc for c in ('y1', 'ymin', 'top'))
    has_x2 = any(c in header_lc for c in ('x2', 'xmax', 'right'))
    has_y2 = any(c in header_lc for c in ('y2', 'ymax', 'bottom'))
    has_w  = any(c in header_lc for c in ('w', 'width'))
    has_h  = any(c in header_lc for c in ('h', 'height'))
    if has_x1 and has_y1 and has_x2 and has_y2:
        return 'xyxy'
    if has_w and has_h:
        return 'xywh'
    return 'label_only'


def _col(header, *candidates):
    lower = [h.lower().strip() for h in header]
    for c in candidates:
        if c in lower:
            return lower.index(c)
    return None


def parse_tsv(tsv_path):
    records = []
    with open(tsv_path, newline='', encoding='utf-8') as f:
        reader = csv.reader(f, delimiter='\t')
        rows = list(reader)
    if not rows:
        return records

    first = rows[0]
    looks_like_header = any(
        not tok.replace('.','').replace('-','').replace('_','').isdigit()
        for tok in first[1:]
    )
    if looks_like_header:
        header, data_rows = first, rows[1:]
    else:
        n = len(first)
        if n >= 6:
            header = ['image_name','label','x1','y1','x2','y2']
        elif n >= 5:
            header = ['image_name','label','x','y','width','height'][:n]
        else:
            header = ['image_name','label']
        data_rows = rows

    variant   = _sniff_columns(header)
    img_col   = _col(header, 'image_name','filename','file_name','image','name') or 0
    label_col = _col(header, 'label','class','category','class_name','classname')

    if variant == 'xyxy':
        x1_col = _col(header,'x1','xmin','left')
        y1_col = _col(header,'y1','ymin','top')
        x2_col = _col(header,'x2','xmax','right')
        y2_col = _col(header,'y2','ymax','bottom')
    elif variant == 'xywh':
        x_col = _col(header,'x','x1','xmin','left')
        y_col = _col(header,'y','y1','ymin','top')
        w_col = _col(header,'w','width')
        h_col = _col(header,'h','height')

    def _safe_float(row, idx):
        if idx is None or idx >= len(row):
            return None
        try:
            return float(row[idx])
        except (ValueError, TypeError):
            return None

    for row in data_rows:
        if not row:
            continue
        rec = {
            'image_name': row[img_col].strip() if img_col < len(row) else '',
            'label':      row[label_col].strip() if label_col is not None and label_col < len(row) else None,
            'x1': None, 'y1': None, 'x2': None, 'y2': None,
        }
        if variant == 'xyxy':
            rec['x1'] = _safe_float(row, x1_col)
            rec['y1'] = _safe_float(row, y1_col)
            rec['x2'] = _safe_float(row, x2_col)
            rec['y2'] = _safe_float(row, y2_col)
        elif variant == 'xywh':
            x = _safe_float(row, x_col)
            y = _safe_float(row, y_col)
            w = _safe_float(row, w_col)
            h = _safe_float(row, h_col)
            if None not in (x, y, w, h):
                rec['x1'], rec['y1'] = x, y
                rec['x2'], rec['y2'] = x + w, y + h
        records.append(rec)
    return records


def build_index(image_dir, annotation_dir):
    img_exts = {'.jpg','.jpeg','.png','.tif','.tiff','.bmp'}
    img_map = {}
    for fname in sorted(os.listdir(image_dir)):
        base, ext = os.path.splitext(fname)
        if ext.lower() in img_exts:
            img_map[base] = os.path.join(image_dir, fname)
    index = []
    for base, img_path in img_map.items():
        tsv_path = os.path.join(annotation_dir, base + '.tsv')
        annots = parse_tsv(tsv_path) if os.path.exists(tsv_path) else []
        index.append({'image_path': img_path, 'annotations': annots})
    return index


def get_grid_cell(x1, y1, x2, y2, img_w, img_h, grid_cols=4, grid_rows=4):
    cx  = (x1 + x2) / 2.0
    cy  = (y1 + y2) / 2.0
    col = int(cx / img_w * grid_cols)
    row = int(cy / img_h * grid_rows)
    col = min(col, grid_cols - 1)
    row = min(row, grid_rows - 1)
    return row * grid_cols + col


def cell_to_bbox(cell, img_w, img_h, grid_cols=4, grid_rows=4):
    row = cell // grid_cols
    col = cell  % grid_cols
    cw  = img_w // grid_cols
    ch  = img_h // grid_rows
    x1  = col * cw
    y1  = row * ch
    return x1, y1, x1 + cw, y1 + ch


def iou(box_a, box_b):
    ix1 = max(box_a[0], box_b[0])
    iy1 = max(box_a[1], box_b[1])
    ix2 = min(box_a[2], box_b[2])
    iy2 = min(box_a[3], box_b[3])
    inter = max(0.0, ix2 - ix1) * max(0.0, iy2 - iy1)
    area_a = (box_a[2] - box_a[0]) * (box_a[3] - box_a[1])
    area_b = (box_b[2] - box_b[0]) * (box_b[3] - box_b[1])
    union  = area_a + area_b - inter
    return inter / union if union > 0 else 0.0

In [ ]:
%%writefile /content/mesad_histogym/mesad_env.py
"""
MESADEnv — HistoGym-style Gymnasium environment for the MESAD dataset.
"""

import os
import random
from typing import Optional

import cv2
import numpy as np
import gymnasium as gym
from gymnasium import spaces

from utils.mesad_annotation import build_index, cell_to_bbox, iou

GRID_COLS  = 4
GRID_ROWS  = 4
OBS_W      = 128
OBS_H      = 128
MAX_STEPS  = 20
IOU_THRESH = 0.3

UP = 0; DOWN = 1; LEFT = 2; RIGHT = 3; SELECT = 4


class MESADEnv(gym.Env):
    metadata = {'render_modes': ['human', 'rgb_array']}
    UP = UP; DOWN = DOWN; LEFT = LEFT; RIGHT = RIGHT; SELECT = SELECT

    def __init__(self, image_dir, annotation_dir,
                 grid_cols=GRID_COLS, grid_rows=GRID_ROWS,
                 obs_w=OBS_W, obs_h=OBS_H,
                 max_steps=MAX_STEPS, iou_thresh=IOU_THRESH,
                 seed=None):
        super().__init__()
        self.image_dir      = image_dir
        self.annotation_dir = annotation_dir
        self.grid_cols      = grid_cols
        self.grid_rows      = grid_rows
        self.obs_w          = obs_w
        self.obs_h          = obs_h
        self.max_steps      = max_steps
        self.iou_thresh     = iou_thresh

        self.index = build_index(image_dir, annotation_dir)
        if not self.index:
            raise FileNotFoundError(f'No images found in {image_dir!r}')

        self.action_space = spaces.Discrete(5)
        self.observation_space = spaces.Box(
            low=0, high=255, shape=(obs_h, obs_w, 3), dtype=np.uint8)

        self._image = None
        self._img_h = self._img_w = 0
        self._annotations = []
        self._cursor_col = self._cursor_row = 0
        self._step_count = 0
        self._current_entry = None

    def reset(self, seed=None, options=None):
        super().reset(seed=seed)
        entry = random.choice(self.index)
        self._current_entry = entry
        img_bgr = cv2.imread(entry['image_path'])
        if img_bgr is None:
            raise IOError(f'Cannot read: {entry["image_path"]}')
        self._image  = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2RGB)
        self._img_h, self._img_w = self._image.shape[:2]
        self._annotations = entry['annotations']
        self._cursor_col  = self.grid_cols // 2
        self._cursor_row  = self.grid_rows // 2
        self._step_count  = 0
        return self._get_obs(), self._get_info()

    def step(self, action):
        terminated = truncated = False
        reward = 0.0
        if action == SELECT:
            reward = self._select_reward()
            terminated = True
        else:
            if action == UP:    self._cursor_row = max(0, self._cursor_row - 1)
            elif action == DOWN:  self._cursor_row = min(self.grid_rows - 1, self._cursor_row + 1)
            elif action == LEFT:  self._cursor_col = max(0, self._cursor_col - 1)
            elif action == RIGHT: self._cursor_col = min(self.grid_cols - 1, self._cursor_col + 1)
            reward = -0.05
        self._step_count += 1
        if self._step_count >= self.max_steps and not terminated:
            reward = -2.0; truncated = True
        return self._get_obs(), reward, terminated, truncated, self._get_info()

    def render(self, mode='human'):
        if self._image is None: return
        canvas = self._image.copy()
        cw = self._img_w // self.grid_cols
        ch = self._img_h // self.grid_rows
        for c in range(1, self.grid_cols):
            cv2.line(canvas, (c*cw, 0), (c*cw, self._img_h), (100,180,255), 1)
        for r in range(1, self.grid_rows):
            cv2.line(canvas, (0, r*ch), (self._img_w, r*ch), (100,180,255), 1)
        x1,y1,x2,y2 = cell_to_bbox(self.cursor_cell, self._img_w, self._img_h,
                                    self.grid_cols, self.grid_rows)
        cv2.rectangle(canvas, (x1,y1), (x2,y2), (0,255,255), 3)
        if mode == 'rgb_array': return canvas

    def close(self): cv2.destroyAllWindows()

    def _get_obs(self):
        if self._image is None:
            return np.zeros((self.obs_h, self.obs_w, 3), dtype=np.uint8)
        x1,y1,x2,y2 = cell_to_bbox(self.cursor_cell, self._img_w, self._img_h,
                                    self.grid_cols, self.grid_rows)
        crop = self._image[y1:y2, x1:x2]
        if crop.size == 0:
            return np.zeros((self.obs_h, self.obs_w, 3), dtype=np.uint8)
        return cv2.resize(crop, (self.obs_w, self.obs_h))

    def _select_reward(self):
        spatial = [a for a in self._annotations
                   if None not in (a['x1'], a['y1'], a['x2'], a['y2'])]
        if not spatial: return 0.0
        box = cell_to_bbox(self.cursor_cell, self._img_w, self._img_h,
                           self.grid_cols, self.grid_rows)
        for a in spatial:
            if iou(box, (a['x1'],a['y1'],a['x2'],a['y2'])) >= self.iou_thresh:
                return 10.0
        return -2.0

    def _get_info(self):
        return {
            'image_path':    self._current_entry['image_path'] if self._current_entry else '',
            'cursor_col':    self._cursor_col,
            'cursor_row':    self._cursor_row,
            'step_count':    self._step_count,
            'n_annotations': len(self._annotations),
        }

    @property
    def cursor_cell(self):
        return self._cursor_row * self.grid_cols + self._cursor_col

## 4. Sanity check the environment

In [ ]:
import sys
sys.path.insert(0, '/content/mesad_histogym')

from mesad_env import MESADEnv
from stable_baselines3.common.env_checker import check_env

env = MESADEnv(
    image_dir      = TRAIN_IMAGES,
    annotation_dir = TRAIN_ANNOTATIONS,
)
print('Dataset size :', len(env.index), 'images')
print('Obs space    :', env.observation_space)
print('Action space :', env.action_space)

check_env(env, warn=True)
print('Environment check PASSED')

## 5. Train  *(change `TOTAL_TIMESTEPS` as needed)*

In [ ]:
from stable_baselines3 import DQN
from stable_baselines3.common.monitor import Monitor
from stable_baselines3.common.callbacks import EvalCallback, CheckpointCallback

# ── Config ────────────────────────────────────────────────────────────────
TOTAL_TIMESTEPS = 50_000   # increase for better performance (try 100k–200k)
MODEL_SAVE_PATH = '/content/dqn_mesad'
GRID_COLS       = 4
GRID_ROWS       = 4
OBS_SIZE        = 128
MAX_STEPS       = 20
SEED            = 42

# ── Environments ──────────────────────────────────────────────────────────
train_env = Monitor(MESADEnv(
    image_dir=TRAIN_IMAGES, annotation_dir=TRAIN_ANNOTATIONS,
    grid_cols=GRID_COLS, grid_rows=GRID_ROWS,
    obs_w=OBS_SIZE, obs_h=OBS_SIZE, max_steps=MAX_STEPS, seed=SEED,
))

val_env = Monitor(MESADEnv(
    image_dir=VAL_IMAGES, annotation_dir=VAL_ANNOTATIONS,
    grid_cols=GRID_COLS, grid_rows=GRID_ROWS,
    obs_w=OBS_SIZE, obs_h=OBS_SIZE, max_steps=MAX_STEPS,
))

# ── Callbacks ─────────────────────────────────────────────────────────────
eval_cb = EvalCallback(
    val_env,
    best_model_save_path='/content/best_model/',
    log_path='/content/best_model/',
    eval_freq=5_000,
    n_eval_episodes=50,
    deterministic=True,
    verbose=1,
)
ckpt_cb = CheckpointCallback(
    save_freq=10_000,
    save_path='/content/checkpoints/',
    name_prefix='dqn_mesad',
)

# ── Model ─────────────────────────────────────────────────────────────────
model = DQN(
    policy                 = 'CnnPolicy',
    env                    = train_env,
    learning_rate          = 1e-4,
    buffer_size            = 10_000,
    batch_size             = 64,
    train_freq             = 4,
    target_update_interval = 1_000,
    exploration_fraction   = 0.2,
    exploration_final_eps  = 0.05,
    verbose                = 1,
    seed                   = SEED,
    tensorboard_log        = '/content/tb_logs/',
)

print(f'Training for {TOTAL_TIMESTEPS:,} steps …')
model.learn(total_timesteps=TOTAL_TIMESTEPS, callback=[eval_cb, ckpt_cb])

model.save(MODEL_SAVE_PATH)
print(f'Model saved to {MODEL_SAVE_PATH}.zip')

## 6. Quick evaluation on the validation set

In [ ]:
from stable_baselines3.common.evaluation import evaluate_policy
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score
import numpy as np

# Use the best model if available, otherwise the final one
best = '/content/best_model/best_model'
import os
load_path = best if os.path.exists(best + '.zip') else MODEL_SAVE_PATH
eval_model = DQN.load(load_path, env=val_env)
print(f'Evaluating: {load_path}')

N_EVAL = 200
y_true, y_pred, rewards, steps_list = [], [], [], []

for _ in range(N_EVAL):
    obs, info = val_env.reset()
    ep_r, n_steps, done = 0.0, 0, False
    while not done:
        action, _ = eval_model.predict(obs, deterministic=True)
        obs, r, term, trunc, info = val_env.step(int(action))
        ep_r += r; n_steps += 1
        done = term or trunc
    rewards.append(ep_r)
    steps_list.append(n_steps)
    has_annot = info['n_annotations'] > 0
    hit        = ep_r >= 10.0
    y_true.append(1 if has_annot else 0)
    y_pred.append(1 if hit       else 0)

print(f'Mean reward : {np.mean(rewards):.3f} ± {np.std(rewards):.3f}')
print(f'Hit-rate    : {np.mean([r >= 10 for r in rewards])*100:.1f}%')
print(f'Mean steps  : {np.mean(steps_list):.1f}')
print(f'Accuracy    : {accuracy_score(y_true, y_pred):.4f}')
print(f'Precision   : {precision_score(y_true, y_pred, zero_division=0):.4f}')
print(f'Recall      : {recall_score(y_true, y_pred, zero_division=0):.4f}')
print(f'F1 Score    : {f1_score(y_true, y_pred, zero_division=0):.4f}')

## 7. Download the trained model

This downloads `dqn_mesad.zip` to your computer.  
Upload it to your Replit project and run:
```bash
cd mesad_histogym
python predict.py --model_path ../dqn_mesad \
    --image_dir ../mesad-real/val/images \
    --annot_dir ../mesad-real/val/annotations
```

In [ ]:
from google.colab import files

# Download the final model
files.download(MODEL_SAVE_PATH + '.zip')

# Also download the best model if it exists
best_zip = '/content/best_model/best_model.zip'
if os.path.exists(best_zip):
    files.download(best_zip)
    print('Also downloaded best_model.zip')

print('Done! Upload the .zip file to your Replit project.')